In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (  
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
    )
from sklearn.model_selection import GridSearchCV , StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer


c:\Users\sanke\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df = pd.read_csv('email.csv')

In [3]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
x = df.drop(columns=['Category'])
y = df['Category']

In [5]:
y.head()

0     ham
1     ham
2    spam
3     ham
4     ham
Name: Category, dtype: str

In [6]:
x.shape

(5573, 1)

In [7]:
x_train , x_test , y_train , y_test = train_test_split(x, y , test_size=0.2 ,random_state=42  )

In [8]:
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

x_train shape: (4458, 1)
y_train shape: (4458,)


In [9]:
x_train = x_train.iloc[:, 0].astype(str)
x_test = x_test.iloc[:, 0].astype(str)

tfidf = TfidfVectorizer(max_features=5000)
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [10]:
print("x_train shape:", x_train.shape)
print("x_train_tfidf shape:", x_train_tfidf.shape)
print("y_train shape:", y_train.shape)

x_train shape: (4458,)
x_train_tfidf shape: (4458, 5000)
y_train shape: (4458,)


In [11]:
cv = StratifiedKFold(
    n_splits=5 ,
    shuffle=True , 
    random_state=42
)
#tunning
param_grid = {
    "n_estimators": [200 ,300,400],
    "max_depth": [None, 10, 20, 30],
    "min_samples_leaf": [1, 2, 4 , 5],
    "max_features": ["sqrt", "log2", 0.5]
}

In [12]:
model = RandomForestClassifier(
    n_estimators=200,
    #update requiered
    random_state=42 , 
    class_weight='balanced',
    n_jobs=-1
)

gridsearch = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

gridsearch.fit(x_train_tfidf , y_train)
best_model = gridsearch.best_estimator_

Fitting 5 folds for each of 144 candidates, totalling 720 fits


c:\Users\sanke\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [13]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("model", best_model)
])

pipeline.fit(
    x_train.astype(str),
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['ham','spam','{""mode"":""full""']"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",5000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None


In [14]:
y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:,-1]

In [15]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label="spam"))
print("Recall:", recall_score(y_test, y_pred, pos_label="spam"))
print("F1 Score:", f1_score(y_test, y_pred, pos_label="spam"))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Not Spam", "Spam"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9829596412556054
Precision: 0.9928571428571429
Recall: 0.8853503184713376
F1 Score: 0.936026936026936

Classification Report:
              precision    recall  f1-score   support

    Not Spam       0.98      1.00      0.99       958
        Spam       0.99      0.89      0.94       157

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
[[957   1]
 [ 18 139]]


In [16]:
joblib.dump(pipeline, r"C:\Users\sanke\OneDrive\Desktop\unclassified\spam_model.joblib")

print("Model saved successfully!")

Model saved successfully!
